In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'huggingface-hub>=0.26.0',
    'python-dotenv>=1.0.0',
    'pyyaml>=6.0',
    'requests>=2.32.0',
    'soundfile>=0.12.1',
    'numpy>=1.26.0',
    'torch>=2.3.0',
    'torchaudio>=2.3.0',
    'transformers>=4.40.0',
    'moshi>=0.1.0',
], check=True)
subprocess.run(['apt-get', 'install', '-qq', '-y', 'ffmpeg'], check=True)

In [ ]:
import os
import json
import time
import threading
from pathlib import Path
from datetime import datetime

import yaml
import requests
import numpy as np
import soundfile as sf
import torch
from huggingface_hub import HfApi

WORK_DIR        = Path('/kaggle/working')
TOKENS_DIR      = WORK_DIR / 'tokens'
CHECKPOINT_PATH = WORK_DIR / 'checkpoint_p2b.json'
CONFIG_DIR      = Path('/kaggle/input/datasets/mirza176528/s2s-pipline-v2-0-2/config')

TOKENS_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR   = 24000
SAVE_EVERY  = 100
DEVICE      = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'[config] device={DEVICE}')

In [ ]:
def load_secrets():
    try:
        from kaggle_secrets import UserSecretsClient
        c = UserSecretsClient()
        secrets = {
            'HF_TOKEN_PRIMARY':   c.get_secret('HF_TOKEN_PRIMARY'),
            'HF_TOKEN_SECONDARY': c.get_secret('HF_TOKEN_SECONDARY'),
            'HF_TOKEN_TERTIARY':  c.get_secret('HF_TOKEN_TERTIARY'),
            'GEMINI_API_KEY':  c.get_secret('GEMINI_API_KEY'),
        }
        print('[secrets] loaded from Kaggle Secrets')
        return secrets
    except Exception:
        pass
    env_file = Path('.env')
    if env_file.exists():
        from dotenv import load_dotenv
        load_dotenv(env_file)
        print('[secrets] loaded from .env')
    required = ['HF_TOKEN_PRIMARY', 'HF_TOKEN_SECONDARY', 'HF_TOKEN_TERTIARY']
    missing = [k for k in required if not os.environ.get(k)]
    if missing:
        raise RuntimeError(f'Missing secrets: {missing}')
    return {k: os.environ[k] for k in required}

SECRETS     = load_secrets()
HF_TOKEN    = SECRETS['HF_TOKEN_PRIMARY']

with open(CONFIG_DIR / 'hf_repos.yaml') as f:
    repos_cfg = yaml.safe_load(f)

STAGE0_REPO = repos_cfg['repos']['stage0_codec']['repo_id']
STAGE1_REPO = repos_cfg['repos']['stage1_ce']['repo_id']
HF_API      = HfApi(token=HF_TOKEN)
print(f'[config] stage0: {STAGE0_REPO}')
print(f'[config] stage1: {STAGE1_REPO}')

In [ ]:
def load_checkpoint():
    if CHECKPOINT_PATH.exists():
        try:
            with open(CHECKPOINT_PATH) as f:
                state = json.load(f)
            print(f'[checkpoint] local — encoded={state["stats"]["encoded"]}')
            return state
        except Exception:
            pass
    try:
        url = f'https://huggingface.co/datasets/{STAGE1_REPO}/resolve/main/checkpoint_p2b.json'
        r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=30)
        if r.status_code == 200:
            state = r.json()
            with open(CHECKPOINT_PATH, 'w') as f:
                json.dump(state, f)
            print(f'[checkpoint] HF fallback — encoded={state["stats"]["encoded"]}')
            return state
    except Exception:
        pass
    print('[checkpoint] fresh start')
    return {
        'done_ids': [],
        'stats': {'encoded': 0, 'failed': 0, 'audio_missing': 0},
        'last_updated': None,
    }


cp_lock = threading.Lock()

def save_checkpoint(state, upload=False):
    with cp_lock:
        state['last_updated'] = datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ')
        tmp = str(CHECKPOINT_PATH) + '.tmp'
        with open(tmp, 'w') as f:
            json.dump(state, f)
        os.replace(tmp, str(CHECKPOINT_PATH))
    if not upload:
        return
    for attempt in range(6):
        try:
            HF_API.upload_file(
                path_or_fileobj=json.dumps(state).encode(),
                path_in_repo='checkpoint_p2b.json',
                repo_id=STAGE1_REPO,
                repo_type='dataset',
                commit_message='p2b checkpoint',
            )
            return
        except Exception:
            time.sleep(min(2 ** attempt, 60))


state    = load_checkpoint()
done_set = set(state['done_ids'])

In [ ]:
print('[mimi] loading frozen Mimi encoder from stage0 checkpoint...')

mimi_ckpt_dir = WORK_DIR / 'mimi_checkpoint'

if not mimi_ckpt_dir.exists():
    print('[mimi] downloading checkpoint from HF stage0...')
    mimi_ckpt_dir.mkdir(parents=True, exist_ok=True)
    for fname in ['config.json', 'model.safetensors']:
        url = f'https://huggingface.co/datasets/{STAGE0_REPO}/resolve/main/mimi_checkpoint/{fname}'
        for attempt in range(6):
            try:
                r = requests.get(url, headers={'Authorization': f'Bearer {HF_TOKEN}'}, timeout=120, stream=True)
                r.raise_for_status()
                with open(mimi_ckpt_dir / fname, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=65536):
                        f.write(chunk)
                print(f'[mimi] downloaded {fname}')
                break
            except Exception as e:
                time.sleep(min(2 ** attempt, 60))

from moshi.models import loaders
mimi_model = loaders.get_mimi(str(mimi_ckpt_dir), device=DEVICE)
mimi_model.eval()
for param in mimi_model.parameters():
    param.requires_grad = False

print(f'[mimi] loaded and frozen on {DEVICE}')


def encode_wav(wav_path):
    audio, sr = sf.read(str(wav_path), dtype='float32')
    if sr != TARGET_SR:
        raise ValueError(f'Expected {TARGET_SR}Hz got {sr}Hz')
    audio_tensor = torch.FloatTensor(audio).unsqueeze(0).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        codes = mimi_model.encode(audio_tensor)
    return codes.squeeze(0).cpu().numpy().T

In [ ]:
labeled_path = WORK_DIR / 'intent_labels.jsonl'
audio_dir    = WORK_DIR / 'clean_final'

all_records  = []
with open(labeled_path, encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            all_records.append(json.loads(line))

pending = [r for r in all_records if r['id'] not in done_set]
print(f'[p2b] total labeled={len(all_records)} already encoded={len(done_set)} pending={len(pending)}')

In [ ]:
updated_records = []

for idx, record in enumerate(pending):
    seg_id    = record['id']
    wav_path  = audio_dir / f'{seg_id}.wav'

    if not wav_path.exists():
        with cp_lock:
            state['stats']['audio_missing'] += 1
            done_set.add(seg_id)
            state['done_ids'].append(seg_id)
        continue

    try:
        tokens     = encode_wav(wav_path)
        token_path = TOKENS_DIR / f'{seg_id}.npy'
        np.save(str(token_path), tokens)

        record['audio_ref']['token_shape'] = list(tokens.shape)
        updated_records.append(record)

        with cp_lock:
            state['stats']['encoded'] += 1
            done_set.add(seg_id)
            state['done_ids'].append(seg_id)

    except Exception as e:
        print(f'  [error] {seg_id}: {e}')
        with cp_lock:
            state['stats']['failed'] += 1
            done_set.add(seg_id)
            state['done_ids'].append(seg_id)

    if (idx + 1) % SAVE_EVERY == 0 or idx + 1 == len(pending):
        upload_now = (idx + 1) % (SAVE_EVERY * 5) == 0
        save_checkpoint(state, upload=upload_now)
        print(f'  [{idx+1}/{len(pending)}] encoded={state["stats"]["encoded"]} '
              f'failed={state["stats"]["failed"]} missing={state["stats"]["audio_missing"]}')

updated_labels_path = WORK_DIR / 'intent_labels_with_shapes.jsonl'
with open(updated_labels_path, 'w', encoding='utf-8') as f:
    for rec in updated_records:
        f.write(json.dumps(rec, ensure_ascii=False) + '\n')

print(f'\n[p2b] encoded {state["stats"]["encoded"]} segments')
print(f'[p2b] token files in: {TOKENS_DIR}')
save_checkpoint(state, upload=True)
print('[done] ready for p2c_upload.ipynb')